# Bankruptcy Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `bankrupt`

## 2 · Project Overview

We predict whether a company will go **bankrupt** based on financial ratios including debt ratio, current ratio, ROE, net income margin, asset turnover, and interest coverage.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a company's financial ratios, predict whether it will go bankrupt.

## 5 · Why This Project Matters

- **Early warning** helps creditors, investors, and regulators.
- Bankruptcy prediction is a classic finance ML problem (Altman Z-score legacy).
- Imbalanced classification with high cost of false negatives.
- Understanding financial health drivers is directly actionable.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 5,000 |
| **Features** | debt_ratio, current_ratio, roe, net_income_margin, asset_turnover, interest_coverage |
| **Target** | bankrupt (0 = healthy, 1 = bankrupt) |
| **Class balance** | ~85/15 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "bankrupt"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: bankrupt
Save dir: E:\Github\Machine-Learning-Projects\Classification\Bankruptcy Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 5,000 companies with financial ratios and bankruptcy outcome.

In [4]:
np.random.seed(SEED)
n = 5000
debt_ratio = np.round(np.random.beta(2, 5, n) * 100, 2)
current_ratio = np.round(np.random.lognormal(0.5, 0.5, n), 2)
roe = np.round(np.random.normal(10, 15, n), 2)
net_income_margin = np.round(np.random.normal(5, 10, n), 2)
asset_turnover = np.round(np.random.uniform(0.2, 3.0, n), 2)
interest_coverage = np.round(np.random.exponential(5, n), 2)

score = (-0.03 * debt_ratio + 0.2 * current_ratio + 0.02 * roe
         + 0.03 * net_income_margin + 0.1 * asset_turnover
         + 0.05 * interest_coverage + np.random.normal(0, 1, n))
bankrupt = (score < np.percentile(score, 15)).astype(int)

df = pd.DataFrame({"debt_ratio": debt_ratio, "current_ratio": current_ratio,
                    "roe": roe, "net_income_margin": net_income_margin,
                    "asset_turnover": asset_turnover, "interest_coverage": interest_coverage,
                    "bankrupt": bankrupt})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['bankrupt'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (5000, 7)
Class balance:
bankrupt
0    0.85
1    0.15
Name: proportion, dtype: float64


,debt_ratio,current_ratio,roe,net_income_margin,asset_turnover,interest_coverage,bankrupt
0,35.37,0.67,8.41,4.48,1.35,0.82,0
1,24.86,1.43,-10.79,3.98,1.33,13.46,0
2,41.60,1.25,31.73,-13.83,2.49,5.25,1
3,16.00,1.55,-4.02,20.34,0.81,1.91,0
4,55.03,2.40,20.87,-8.68,2.53,16.90,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (5000, 7)

Missing values:
debt_ratio           0
current_ratio        0
roe                  0
net_income_margin    0
asset_turnover       0
interest_coverage    0
bankrupt             0
dtype: int64

Duplicate rows: 0

Dtypes:
debt_ratio           float64
current_ratio        float64
roe                  float64
net_income_margin    float64
asset_turnover       float64
interest_coverage    float64
bankrupt               int64
dtype: object

Target 'bankrupt' confirmed. Value counts:
bankrupt
0    4250
1     750
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Bankruptcy Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `bankrupt`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Bankruptcy Distribution")
ax.set_xticklabels(["Healthy (0)", "Bankrupt (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Class balance: {dict(df[TARGET].value_counts())}")
print(f"Bankruptcy rate: {df[TARGET].mean():.1%}")

Class balance: {0: np.int64(4250), 1: np.int64(750)}
Bankruptcy rate: 15.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4000, 6), Test: (1000, 6)
Train class distribution:
bankrupt
0    0.85
1    0.15
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None in synthetic data.
- **Categorical**: None — all numeric financial ratios.
- **Scaling**: Not needed for tree-based models.
- **Class imbalance**: ~15% bankrupt — moderate imbalance, use stratified split.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["altman_proxy"] = (1.2 * X_train["current_ratio"] + 0.6 * X_train["roe"] / 100
                           + 3.3 * X_train["net_income_margin"] / 100 + X_train["asset_turnover"])
X_test["altman_proxy"] = (1.2 * X_test["current_ratio"] + 0.6 * X_test["roe"] / 100
                          + 3.3 * X_test["net_income_margin"] / 100 + X_test["asset_turnover"])

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (7): ['debt_ratio', 'current_ratio', 'roe', 'net_income_margin', 'asset_turnover', 'interest_coverage', 'altman_proxy']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.8510
F1       : 0.3196

              precision    recall  f1-score   support

           0       0.88      0.96      0.92       850
           1       0.51      0.23      0.32       150

    accuracy                           0.85      1000
   macro avg       0.69      0.60      0.62      1000
weighted avg       0.82      0.85      0.83      1000



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                      
NearestCentroid                 0.697           0.711961  0.779020  0.738724   0.840009   0.697    0.015877
GaussianNB                      0.843           0.619412  0.764722  0.828529   0.820750   0.843    0.015193
ExtraTreeClassifier             0.782           0.602745  0.602745  0.788018   0.794745   0.782    0.022514
LogisticRegression              0.851           0.596667  0.784933  0.826833   0.821092   0.851    0.017734
LinearDiscriminantAnalysis      0.848           0.594902  0.785396  0.824631   0.817583   0.848    0.017076
LabelPropagation                0.788           0.592549  0.670737  0.789703   0.791460   0.788    0.343640
AdaBoostClassifier              0.842           0.591373  0.771933  0.820263   0.811288   0.842    0.2

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.8130
F1: 0.2835


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.2899  (1.3s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[40]	valid_0's binary_logloss: 0.368139
LightGBM F1: 0.2176  (0.7s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression     0.851  0.3196     0.5072  0.2333
CatBoost                0.853  0.2899     0.5263  0.2000
FLAML                   0.813  0.2835     0.3333  0.2467
LightGBM                0.849  0.2176     0.4884  0.1400

Top 3 models: ['Logistic Regression', 'CatBoost', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.8510
    F1       : 0.3196
    Precision: 0.5072
    Recall   : 0.2333

  CatBoost:
    Accuracy : 0.8530
    F1       : 0.2899
    Precision: 0.5263
    Recall   : 0.2000

  FLAML:
    Accuracy : 0.8130
    F1       : 0.2835
    Precision: 0.3333
    Recall   : 0.2467


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       850
           1       0.51      0.23      0.32       150

    accuracy                           0.85      1000
   macro avg       0.69      0.60      0.62      1000
weighted avg       0.82      0.85      0.83      1000


Total errors: 149 / 1000 (14.9%)

Sample misclassifications:
      debt_ratio  current_ratio    roe  net_income_margin  asset_turnover  interest_coverage  altman_proxy  actual  predicted  correct
2086       36.07           1.72   2.90              10.58            0.49               0.69       2.92054       1          0    False
4841       18.95           1.88  -9.58             -17.33            2.88               2.80       4.50663       1          0    False
641        46.74           0.63  11.80               7.73            2.73               6.08       3.81189       1          0    False
3906       68.4

## 25 · Interpretation and Business Insight

**Key findings:**
- **Debt ratio** and **current ratio** are the strongest predictors.
- **Interest coverage** below a threshold signals distress.
- The engineered Altman proxy adds predictive value.

**Business takeaway:** Monitor debt-to-asset ratio and liquidity metrics for early warning signs.

## 26 · Limitations

1. Synthetic data with simplified relationships.
2. No industry sector or macroeconomic context.
3. Missing qualitative factors (management quality, market position).
4. Static snapshot — no time-series financial trajectory.
5. ~15% bankruptcy rate is higher than real-world base rates.

## 27 · How to Improve This Project

1. Use real financial statement data (SEC EDGAR filings).
2. Add time-series of ratios over multiple quarters.
3. Include industry sector as a feature.
4. Try Altman Z-score as a stronger engineered feature.
5. Calibrate probabilities for risk scoring.

## 28 · Production Considerations

- Retrain quarterly with updated financial data.
- Output calibrated probabilities, not binary labels.
- Flag high-risk companies for manual review.
- Monitor for sector-wide drift (recessions).

## 29 · Common Mistakes

1. Ignoring class imbalance — accuracy misleads.
2. Not considering cost of false negatives (missed bankruptcies).
3. Using only accuracy without precision-recall analysis.
4. Overfitting on small bankrupt class.
5. Not validating on out-of-time data.

## 30 · Mini Challenge / Exercises

1. Change bankruptcy threshold to 10% — how does class imbalance affect F1?
2. Try `class_weight='balanced'` in Logistic Regression.
3. Calculate the Altman Z-score properly and compare to model predictions.
4. Plot PR curve and find the optimal threshold.
5. Remove `altman_proxy` — does performance drop?

## 31 · Final Summary / Key Takeaways

1. **Financial ratios** are strong bankruptcy predictors.
2. **Class imbalance** requires careful metric selection (F1, PR-AUC).
3. **Engineered features** (Altman proxy) add value.
4. **Tree-based models** handle ratio features well.
5. **Cost-sensitive evaluation** matters — missing bankruptcies is costly.
6. **Real-world deployment** needs calibrated probabilities and human oversight.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Bankruptcy Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.853,
    "f1": 0.2899,
    "precision": 0.5263,
    "recall": 0.2
  },
  "LightGBM": {
    "accuracy": 0.849,
    "f1": 0.2176,
    "precision": 0.4884,
    "recall": 0.14
  },
  "Logistic Regression": {
    "accuracy": 0.851,
    "f1": 0.3196,
    "precision": 0.5072,
    "recall": 0.2333
  },
  "FLAML": {
    "accuracy": 0.813,
    "f1": 0.2835,
    "precision": 0.3333,
    "recall": 0.2467
  }
}
